# 01 — Boosting intuition: focus on the mistakes

*Chapter 07 — AdaBoost · Notebook 1 of 5*

Chapter 06 built an ensemble by training many trees **in parallel** and averaging their equal votes —
the forest's stability came from independence. This chapter takes the opposite road. We train weak
learners **one after another**, and each new one is told to pay attention to the points the ensemble
still gets wrong. That single idea — *focus on the mistakes* — is **boosting**, and AdaBoost is its
first and most transparent member. We build the whole machine by hand.

**Prerequisites.**
- **Chapter 04 (Decision Trees), NB 1–2:** a **decision stump** is a depth-1 tree — one yes/no
  question on one feature. It is a deliberately *weak* learner, and it is all we need here.
- **Chapter 06 (Random Forests), NB 1:** an **ensemble** combines many models by voting; bagging
  trains them independently and gives each an equal vote.
- **Module 00:** the train/test split (NB 04) and accuracy (NB 06).

**What you'll be able to do.**
- Run AdaBoost's **reweighting loop** by hand: sample weights, the weighted error ε, the learner
  weight α, and the update that pushes weight onto the hard cases.
- Explain why each weak learner's vote is weighted by α = ln((1 − ε)/ε).
- Watch a pile of weak stumps **compound** into a strong classifier.
- Recognize that the loop you wrote **is** `AdaBoostClassifier` — matched to machine precision.

## Where we are — from bagging to boosting

Chapter 06's recipe was: draw many independent bootstrap samples, grow a full tree on each, and let
them vote **equally**. Because the trees are trained in parallel and never consult one another, their
errors are partly independent, and averaging cancels the wobble — variance goes down.

Boosting reverses every word of that. The learners are trained **in sequence**, each one **depends**
on the ones before it (it studies their mistakes), and their votes are **unequal** (a better learner
speaks louder). Bagging asks many independent opinions and averages them; boosting builds one opinion,
sees exactly where it errs, and corrects — round after round.

| | Bagging (ch 06) | Boosting (this chapter) |
|---|---|---|
| order | parallel, independent | sequential, dependent |
| each learner sees | its own bootstrap sample | reweighted data — last round's mistakes |
| votes | equal | weighted by α |
| mainly cures | variance | bias **and** variance |

We stay on a two-dimensional set — two interleaving crescents — so we can *draw* every step.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import CLASS_CYCLE, COLORS

viz.use_course_style()
SEED = 0

# Two interleaving crescents. Two dimensions let us *draw* every step of the
# loop; the moderate noise keeps a single stump honestly weak.
X, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
n_train = X_train.shape[0]
print(f"train: {n_train} points, classes {np.bincount(y_train).tolist()}")
print(f"test:  {X_test.shape[0]} points")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for cls in (0, 1):
    m = y_train == cls
    ax.scatter(
        X_train[m, 0], X_train[m, 1], color=CLASS_CYCLE[cls],
        edgecolor=COLORS["text"], linewidth=0.6, s=45, label=f"class {cls}",
    )
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend(loc="best")
plt.show()

**Read the data.** Two interleaving crescents, 140 points each — balanced classes. No single
straight line, and no single axis-aligned cut, can separate them *without error*: the boundary
genuinely curves. That is the point. One weak learner will fall short here, which is exactly the
setting where boosting earns its keep.

## A weak learner: the decision stump

Our building block is the simplest tree there is. A **decision stump** is a tree of depth 1: it asks
one question — "is feature *j* above a threshold?" — and predicts one class on each side. We met it in
chapter 04 as the first split a tree ever makes. On a curved problem like this one it cannot do much;
it is a **weak learner**, barely better than a coin flip. Boosting's wager is that many weak learners,
each aimed at the last one's mistakes, can together be strong.

In [ ]:
stump = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(X_train, y_train)
print(f"single stump: train accuracy = {stump.score(X_train, y_train):.4f}")
print(f"              test  accuracy = {stump.score(X_test, y_test):.4f}")

viz.plot_decision_boundary(stump, X_train, y_train)
plt.show()

**Read the figure.** The stump draws one straight cut and colours each side with a single
class. It gets about **87%** of the test points right — not bad, but look at the misses: an entire arc
of one crescent falls on the wrong side, because one straight line cannot follow a curve. This is our
weak learner. The rest of the notebook turns a sequence of these into something that *can* follow the
curve.

## The idea: reweight, then repeat

Here is the whole AdaBoost loop in words, before any algebra.

1. Give every training point a **weight**. Start them equal.
2. Fit a weak learner that minimises the **weighted** error — it tries hardest on the heaviest points.
3. Measure its weighted error **ε** (the fraction of weight it gets wrong), and give the learner a
   **vote weight α** — large when ε is small, zero when ε = ½ (a coin flip earns no say).
4. **Increase** the weight on every point this learner got wrong, and decrease it on the rest.
5. Go back to step 2. The next learner now faces a reshaped problem, tilted toward the hard cases.

After *T* rounds the ensemble predicts by a **weighted vote**: each learner votes its class, scaled by
its α, and we take the sign of the sum. We build steps 1–4 by hand now; the weighted vote is the
subject of notebook 2.

In [ ]:
# Boosting works in labels {-1, +1}; start every point equally important.
y_pm = np.where(y_train == 1, 1, -1)
w = np.full(n_train, 1.0 / n_train)

# Round 1: fit a stump to the WEIGHTED data, then read its weighted error.
stump1 = DecisionTreeClassifier(max_depth=1, random_state=SEED)
stump1.fit(X_train, y_train, sample_weight=w)
pred1 = np.where(stump1.predict(X_train) == 1, 1, -1)
miss1 = pred1 != y_pm
eps1 = w[miss1].sum() / w.sum()
alpha1 = np.log((1 - eps1) / eps1)
print(f"round 1: weighted error  eps   = {eps1:.4f}")
print(f"         vote weight      alpha = ln((1-eps)/eps) = {alpha1:.4f}")
print(f"         misclassifies {int(miss1.sum())} of {n_train} training points")

**Read the result.** The first stump gets a weighted error of **ε ≈ 0.157** — better than a
coin flip (ε = 0.5), so its vote weight **α ≈ 1.68** is comfortably positive. The formula
α = ln((1 − ε)/ε) has exactly the behaviour we want: as ε → 0 the learner is trusted more and more
(α → ∞), and at ε = ½ it earns **α = 0** — no vote at all, because a coin flip carries no information.

One honest note on conventions: this α = ln((1 − ε)/ε) is the form scikit-learn's AdaBoost uses (its
"SAMME" algorithm). The classic 1997 formula carries an extra ½ out front; it gives the **same**
classifier, and we'll see precisely why in notebook 2.

In [ ]:
# Up-weight what round 1 got wrong: multiply those weights by exp(alpha), renormalise.
before = w[miss1].sum()
w = w * np.exp(alpha1 * miss1)
w = w / w.sum()
after = w[miss1].sum()
print(f"weight on the {int(miss1.sum())} misclassified points: {before:.4f} -> {after:.4f}")
print(f"({100 * miss1.mean():.0f}% of points now hold {100 * after:.0f}% of the weight)")

**Read the result.** Before reweighting, the 44 misclassified points carried their fair share
of the weight: 16% of the points, 16% of the weight. After one update they carry **half of all the
weight**. The next stump will be graded almost entirely on whether it fixes these specific points —
"focus on the mistakes" is no longer a slogan, it is arithmetic. The points the first stump found easy
have faded into the background.

In [ ]:
# Round 2: a fresh stump on the reweighted data — it now cares most about round 1's misses.
stump2 = DecisionTreeClassifier(max_depth=1, random_state=SEED)
stump2.fit(X_train, y_train, sample_weight=w)
pred2 = np.where(stump2.predict(X_train) == 1, 1, -1)
miss2 = pred2 != y_pm
eps2 = w[miss2].sum() / w.sum()
alpha2 = np.log((1 - eps2) / eps2)
print(f"round 2: weighted error eps   = {eps2:.4f}   (round 1: {eps1:.4f})")
print(f"         vote weight     alpha = {alpha2:.4f}   (round 1: {alpha1:.4f})")

**Read the result.** Round 2's weighted error (**ε ≈ 0.244**) is *higher* than round 1's
(0.157) — and that is not a step backwards. We deliberately reshaped the data to dwell on the hardest
points, so any learner scores worse on it; a higher ε means this round faces a harder problem and so
earns a smaller vote (**α ≈ 1.13** versus 1.68). Each round is a specialist on a different,
progressively harder slice of the data, and α books each specialist's vote at its true worth.

In [ ]:
# Replay rounds 1-3, snapshotting the weights each round SEES and the cut it makes.
w_snap = np.full(n_train, 1.0 / n_train)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for r, ax in zip((1, 2, 3), axes, strict=False):
    h = DecisionTreeClassifier(max_depth=1, random_state=SEED)
    h.fit(X_train, y_train, sample_weight=w_snap)
    sizes = w_snap / w_snap.max() * 300 + 8
    for cls in (0, 1):
        m = y_train == cls
        ax.scatter(
            X_train[m, 0], X_train[m, 1], s=sizes[m], color=CLASS_CYCLE[cls],
            edgecolor=COLORS["text"], linewidth=0.5, alpha=0.9,
        )
    feat = int(h.tree_.feature[0])
    thr = float(h.tree_.threshold[0])
    if feat == 0:
        ax.axvline(thr, color=COLORS["model"], linewidth=2.0, linestyle="--")
    else:
        ax.axhline(thr, color=COLORS["model"], linewidth=2.0, linestyle="--")
    ax.set_title(f"round {r}: cut x{feat + 1} at {thr:.2f}")
    ax.set_xlabel("x1")
    pred = np.where(h.predict(X_train) == 1, 1, -1)
    miss = pred != y_pm
    eps = w_snap[miss].sum() / w_snap.sum()
    a = np.log((1 - eps) / eps)
    w_snap = w_snap * np.exp(a * miss)
    w_snap = w_snap / w_snap.sum()
axes[0].set_ylabel("x2")
fig.suptitle("Point size shows sample weight: it piles onto the hard cases")
plt.show()

**Read the figure.** Point size shows each sample's weight. In round 1 every point is the same
size — the stump sees a flat problem. By round 2 the dots that the first cut got wrong have swelled,
and the second stump places its line to catch them. By round 3 the weight has concentrated further,
along the seam where the two crescents interlock — exactly the region a single straight cut keeps
getting wrong. The committee is not repeating itself; each member is sent to a different hard place.

In [ ]:
def boost(X, y01, T, seed=SEED):
    '''Run T rounds of AdaBoost (SAMME, binary) by hand on labels y01 in {0, 1}.

    Returns the stumps, their vote weights alpha, and the running-ensemble
    training error after each round.
    '''
    y_pm = np.where(y01 == 1, 1, -1)
    n = len(y_pm)
    w = np.full(n, 1.0 / n)
    stumps, alphas = [], []
    F = np.zeros(n)
    train_err = []
    for _ in range(T):
        h = DecisionTreeClassifier(max_depth=1, random_state=seed)
        h.fit(X, y01, sample_weight=w)
        pred = np.where(h.predict(X) == 1, 1, -1)
        miss = pred != y_pm
        eps = min(max(w[miss].sum() / w.sum(), 1e-12), 1 - 1e-12)
        alpha = np.log((1 - eps) / eps)
        w = w * np.exp(alpha * miss)
        w = w / w.sum()
        stumps.append(h)
        alphas.append(alpha)
        # running alpha-weighted vote (NB 2 formalises it); here only to score the ensemble
        F = F + alpha * pred
        train_err.append(float(np.mean(np.sign(F) != y_pm)))
    return stumps, np.array(alphas), np.array(train_err)


T = 50
stumps, alphas, train_err = boost(X_train, y_train, T)
for r in (1, 3, 10, 50):
    print(f"after {r:2d} rounds: ensemble training error = {train_err[r - 1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
rounds = np.arange(1, T + 1)
ax.plot(rounds, train_err, color=COLORS["model"], linewidth=2.2)
ax.scatter([1], [train_err[0]], color=COLORS["highlight"], zorder=5, s=60)
ax.set_xlabel("number of rounds (weak learners)")
ax.set_ylabel("ensemble training error")
ax.set_ylim(0, float(train_err.max()) * 1.1)
plt.show()

**Read the figure.** A single stump sits near 16% training error on its own. Chained the
boosting way, the ensemble's training error falls *overall* — with small wiggles up and down, since it
is the **combined** vote, not any single stump, that improves — reaching about **1.4%** by 50 rounds.
No single stump ever beats ~84% on its own; the *combination* grew strong because each new stump was
aimed where the running ensemble was still wrong.

One caveat to carry forward: this is **training** error. Driving it toward zero is encouraging, but it
is not the same as generalising — and a method that chases the hardest points so relentlessly has a
known weak spot when some of those points are mislabeled. We measure both the promise and the risk in
notebook 3.

In [ ]:
ada = AdaBoostClassifier(n_estimators=T, random_state=SEED)
ada.fit(X_train, y_train)

alpha_diff = float(np.max(np.abs(alphas - ada.estimator_weights_)))
print(f"by-hand alpha[:5]              = {np.round(alphas[:5], 4)}")
print(f"sklearn estimator_weights_[:5] = {np.round(ada.estimator_weights_[:5], 4)}")
print(f"max |alpha difference| over {T} rounds = {alpha_diff:.1e}")

# Same predictions: our running vote vs sklearn on the held-out test set.
y_test_pm = np.where(y_test == 1, 1, -1)
F_test = np.zeros(len(y_test))
for a, h in zip(alphas, stumps, strict=False):
    F_test = F_test + a * np.where(h.predict(X_test) == 1, 1, -1)
hand_acc = float(np.mean(np.sign(F_test) == y_test_pm))
print(f"by-hand test accuracy @ T={T} = {hand_acc:.4f}")
print(f"sklearn test accuracy @ T={T} = {ada.score(X_test, y_test):.4f}")

**Read the result.** The library is not doing anything we did not. `AdaBoostClassifier` produces
the **same** learner weights as our by-hand loop — agreeing to about fifteen decimal places — and the
**same** test predictions (accuracy **0.9417** for both, up from the single stump's 0.867).
`AdaBoostClassifier` is this loop, written in optimised code. You now know what is inside it.

## What we have, and what is still open

You built AdaBoost's engine: weight the points, fit a weak learner, weight its vote by α, push weight
onto the mistakes, repeat. Three honest notes before we go on.

- **The α convention.** We used α = ln((1 − ε)/ε), scikit-learn's SAMME form. The original 1997 AdaBoost
  writes ½ ln((1 − ε)/ε). The factor changes the *scale* of every α but not the sign, so the final
  weighted vote lands on the same side — the same classifier. Notebook 2 derives where this α comes
  from (it is not arbitrary — it falls out of a loss function).
- **Bias and variance.** By chasing the hard cases, boosting lowers **bias** as well as variance — it
  builds an ever more expressive model. That very appetite for hard cases is also its vulnerability: a
  **mislabeled** point looks like an eternally hard case, and the reweighting keeps feeding it more and
  more weight. We meet this head-on in notebooks 3 and 5.
- **Training error is not the whole story.** We drove *training* error down here. Generalisation — and
  AdaBoost's surprising, conditional resistance to overfitting — is notebook 3.

## Your turn

1. **Find the crossover (easy).** In the parity cell, refit `AdaBoostClassifier` with `n_estimators`
   set to 1, then 5, then 20, and record the test accuracy each time. At how few stumps does the
   boosted ensemble first beat the single stump's 0.867? (You are watching weak learners compound.)
2. **Read the α formula (medium).** Compute α = ln((1 − ε)/ε) by hand for ε = 0.01, ε = 0.25, and
   ε = 0.5. Which earns the loudest vote, and which earns none? Explain in one sentence what α = 0 says
   about a learner that is right exactly half the time.
3. **Derive the weight jump (harder).** The update multiplies a misclassified point's weight by exp(α)
   and leaves a correct point's weight unchanged (before renormalising). Using α₁ ≈ 1.68, show that a
   missed point's weight grows by a factor of about **5.4** relative to a correct one, and check that
   against the weights printed in the reweighting cell.

## What you built

- You ran the **AdaBoost reweighting loop** by hand: uniform **sample weights**, a stump fit to the
  **weighted** error, the **weighted error ε**, the **learner weight** α = ln((1 − ε)/ε), and the
  update that multiplies the mistakes' weights by exp(α).
- You saw the weight **concentrate on the hard cases** — the first round's 44 misses went from 16% to
  50% of the total weight in a single step.
- You watched weak learners **compound**: the ensemble's training error fell from 16% to about 1% over
  50 rounds, though no single stump was better than ~84%.
- You confirmed your by-hand loop **is** `AdaBoostClassifier` — identical learner weights (to ~1e-15)
  and identical test accuracy (0.9417).

**Vocabulary you now own:** boosting · weak learner · decision stump · sample weights · weighted error
ε · learner weight α · reweighting · sequential ensemble (vs bagging's parallel one).

## References

- Freund, Y., & Schapire, R. E. (1997). *A decision-theoretic generalization of on-line learning and
  an application to boosting.* Journal of Computer and System Sciences 55(1), 119–139.
  DOI: [10.1006/jcss.1997.1504](https://doi.org/10.1006/jcss.1997.1504)
- Zhu, J., Zou, H., Rosset, S., & Hastie, T. (2009). *Multi-class AdaBoost.* Statistics and Its
  Interface 2(3), 349–360. DOI: [10.4310/SII.2009.v2.n3.a8](https://doi.org/10.4310/SII.2009.v2.n3.a8)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.1 (boosting). DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)

*Previous: Chapter 06 — Random Forests (NB 5: forest cover type). Next: 02 — Weak learners and the
additive model.*